### Data Processing

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
!pip install mlxtend
from mlxtend.plotting import plot_decision_regions 
from sklearn.ensemble import RandomForestClassifier


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [71]:
seattle = pd.read_csv("listings.csv.gz")

In [72]:
# turn price into string
seattle["price"] = (
    seattle["price"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

In [73]:
# filter price <= 0
seattle = seattle[seattle["price"] > 0]
# remove extreme price
p99_seattle = seattle["price"].quantile(0.99)
seattle = seattle[seattle["price"] <= p99_seattle]

In [74]:
# remove extreme value
p99_bedroom = seattle["bedrooms"].quantile(0.99)
p99_beds = seattle["beds"].quantile(0.99)
p99_bathroom = seattle["bathrooms"].quantile(0.99)
p99_accomodates = seattle["accommodates"].quantile(0.99)
seattle = seattle[seattle["bedrooms"] <= p99_bedroom]
seattle = seattle[seattle["beds"] <= p99_beds]
seattle = seattle[seattle["bathrooms"] <= p99_bathroom]
seattle = seattle[seattle["accommodates"] <= p99_accomodates]
seattle.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6606,https://www.airbnb.com/rooms/6606,20250925032813,2025-09-25,city scrape,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","A peaceful yet highly accessible neighborhood,...",https://a0.muscache.com/pictures/45742/21116d7...,14942,...,4.77,4.88,4.57,str-opli-19-002622,f,3,3,0,0,0.82
1,9419,https://www.airbnb.com/rooms/9419,20250925032813,2025-09-25,city scrape,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"Lots of restaurants (see our guide book) bars,...",https://a0.muscache.com/pictures/56645186/e5fb...,30559,...,4.89,4.70,4.69,Exempt,f,10,0,10,0,1.19
3,11012,https://www.airbnb.com/rooms/11012,20250925032813,2025-09-25,city scrape,"the orange house, quiet 'n central",NaN,NaN,https://a0.muscache.com/pictures/682034/54bc27...,14942,...,4.72,4.86,4.74,str-opli-19-002622,f,3,3,0,0,0.51
4,25002,https://www.airbnb.com/rooms/25002,20250925032813,2025-09-25,city scrape,Beautiful Private Spot in North Ballard,"-Great eating , Delancey, Fat Hen, 3 blocks aw...",Great walking neighborhood! We are in between...,https://a0.muscache.com/pictures/491561/cf5270...,102684,...,4.98,4.90,4.90,STR-OPLI-19-002617,t,1,1,0,0,6.06
5,26795,https://www.airbnb.com/rooms/26795,20250925032813,2025-09-25,city scrape,Lake Union Cottage - Shore and City View,"This sunny, corner lot is directly across from...",This area of the Eastlake Neighborhood is quie...,https://a0.muscache.com/pictures/179416/54927c...,114228,...,4.58,4.83,4.36,NaN,f,1,1,0,0,0.35


In [75]:
# create relative price columns
# Relative Pricing =  Pricing / Mean Pricing of listings with same neighborhood, # of bedrooms and bathrooms, and roomtype
group_cols = ["bedrooms", "bathrooms", "neighbourhood_cleansed", "room_type"]
seattle["avg_price_group"] = seattle.groupby(group_cols)["price"].transform("mean")
seattle["relative_price"] = seattle["price"] / seattle["avg_price_group"]

In [76]:
# host response time
mapping_host_response_time = {
    'within an hour': 1,
    'within a few hours': 2,
    'within a day': 3,
    'a few days or more': 4
}

seattle['host_response_time'] = seattle['host_response_time'].map(mapping_host_response_time)

In [77]:
# is_long_term
## minimum nights -> short-term listing vs. long-term listing
seattle['is_long_term'] = (seattle["minimum_nights"] > 7).astype(int)

In [78]:
# recent_Review_proportion : number_of_reviews_ltm / number_of_reviews
seattle['recent_Review_proportion'] = seattle['number_of_reviews_ltm'] / seattle['number_of_reviews']

In [79]:
features = [
    "neighbourhood_group_cleansed",
    "review_scores_rating",
    # relative pricing
    "relative_price",
    # housing characteristics
    "bedrooms",
    "beds",
    "bathrooms",
    "accommodates",
    "room_type",
    "is_long_term",
    # host characteristics
    "host_response_time",
    "host_identity_verified", 
    # others
    "number_of_reviews",
    "recent_Review_proportion",
    "estimated_occupancy_l365d"
]

In [80]:
seattle = seattle[features] 

In [81]:
seattle = seattle.dropna()
seattle

,neighbourhood_group_cleansed,review_scores_rating,relative_price,bedrooms,beds,bathrooms,accommodates,room_type,is_long_term,host_response_time,host_identity_verified,number_of_reviews,recent_Review_proportion,estimated_occupancy_l365d
0,Other neighborhoods,4.60,0.772909,1.0,1.0,1.0,1,Entire home/apt,1,2.0,t,161,0.000000,0
1,Other neighborhoods,4.73,0.984592,1.0,2.0,3.0,2,Private room,0,1.0,t,220,0.063636,84
3,Other neighborhoods,4.79,1.232497,3.0,3.0,2.0,8,Entire home/apt,1,2.0,t,98,0.000000,0
4,Ballard,4.93,0.741860,1.0,2.0,1.0,4,Entire home/apt,0,1.0,t,1139,0.054434,255
5,Cascade,4.54,0.969773,2.0,3.0,1.0,3,Entire home/apt,1,2.0,t,64,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6907,West Seattle,5.00,0.770687,1.0,1.0,1.0,2,Entire home/apt,0,1.0,t,3,1.000000,18
6910,Other neighborhoods,5.00,0.898058,3.0,3.0,3.5,6,Entire home/apt,0,1.0,f,2,1.000000,12
6930,Beacon Hill,5.00,1.124479,2.0,2.0,2.0,4,Entire home/apt,0,1.0,t,1,1.000000,6
6938,Other neighborhoods,4.67,0.952474,1.0,2.0,1.0,3,Entire home/apt,0,1.0,t,3,1.000000,18


In [82]:
%cd imt574_ml_final_project
%pwd

/Users/peiyingw/imt574_ml_final_project


'/Users/peiyingw/imt574_ml_final_project'

In [83]:
# turning pandas to csv 
seattle.to_csv('seattle_filtered.csv', index=False)